# Dashboard de regresión lineal múltiple

Este cuaderno está listo para subir a Google Colab y presentar tu proyecto de regresión.

### Qué incluye
- Carga de los tres datasets
- Entrenamiento de 3 modelos
- Métricas: R², MSE, RMSE, MAE
- Gráficas de dispersión, correlación y real vs predicho
- Resumen final para entregar en clase

### Instrucciones
1. Sube a Colab este archivo .ipynb
2. Suba también los archivos CSV de la carpeta `data/`
3. Ejecuta todas las celdas
4. Si quieres, exporta el notebook como HTML o PDF

In [ ]:
%pip install pandas numpy matplotlib scikit-learn joblib scipy -q

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Si el notebook está en Colab, se pueden subir los CSVs manualmente
if not os.path.exists('data/dolar_data.csv'):
    try:
        from google.colab import files
        uploaded = files.upload()
        print('Archivos subidos:', list(uploaded.keys()))
    except Exception:
        print('No se detectaron archivos CSV. Verifica que tengas la carpeta data/ o súbela manualmente.')

data_dir = 'data' if os.path.exists('data/dolar_data.csv') else '.'
print('Directorio de datos:', data_dir)

In [ ]:
EJERCICIOS = {
    'dolar': {
        'archivo': 'dolar_data.csv',
        'titulo': 'Predicción del precio del dólar',
        'features': ['Dia', 'Inflacion', 'Tasa_interes'],
        'target': 'Precio_Dolar',
        'unidad': 'pesos'
    },
    'glucosa': {
        'archivo': 'glucosa_data.csv',
        'titulo': 'Predicción del nivel de glucosa',
        'features': ['Edad', 'IMC', 'Actividad_Fisica'],
        'target': 'Nivel_Glucosa',
        'unidad': 'mg/dL'
    },
    'energia': {
        'archivo': 'energia_data.csv',
        'titulo': 'Predicción del consumo de energía',
        'features': ['Temperatura', 'Hora', 'Dia_Semana'],
        'target': 'Consumo_Energia',
        'unidad': 'kWh'
    }
}

SEED = 42

def entrenar_modelo(df, features, target):
    X = df[features]
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
    modelo = LinearRegression()
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    metrics = {
        'MSE': mean_squared_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAE': mean_absolute_error(y_test, pred),
        'R2': r2_score(y_test, pred),
    }

    coef = pd.DataFrame({
        'variable': features,
        'coeficiente': modelo.coef_
    })
    return modelo, metrics, coef

resumen = []
for clave, cfg in EJERCICIOS.items():
    path = os.path.join(data_dir, cfg['archivo'])
    if not os.path.exists(path):
        raise FileNotFoundError(f'No existe {path}. Sube el archivo CSV antes de ejecutar esta celda.')
    df = pd.read_csv(path)
    df = df[cfg['features'] + [cfg['target']]].apply(pd.to_numeric, errors='coerce').dropna().drop_duplicates()
    modelo, metrics, coef = entrenar_modelo(df, cfg['features'], cfg['target'])
    resumen.append({
        'modelo': clave,
        'titulo': cfg['titulo'],
        'registros': len(df),
        'R2': metrics['R2'],
        'RMSE': metrics['RMSE'],
        'coeficientes': coef.to_dict('records')
    })
    print(f'\n=== {cfg[titulo]} ===')
    print(pd.DataFrame([metrics]).round(6))
    print('Ecuación:')
    eq = f"{cfg['target']} = {modelo.intercept_:.4f}"
    for c in coef.itertuples(index=False):
        signo = '+' if c.coeficiente >= 0 else '-'
        eq += f" {signo} {abs(c.coeficiente):.4f}·{c.variable}"
    print(eq)

df_resumen = pd.DataFrame([{
    'Modelo': r['modelo'],
    'Título': r['titulo'],
    'Registros': r['registros'],
    'R²': round(r['R2'], 4),
    'RMSE': round(r['RMSE'], 4)
} for r in resumen])
display(df_resumen.style.set_caption('Resumen general de los modelos'))

In [ ]:
for clave, cfg in EJERCICIOS.items():
    df = pd.read_csv(os.path.join(data_dir, cfg['archivo']))
    df = df[cfg['features'] + [cfg['target']]].apply(pd.to_numeric, errors='coerce').dropna().drop_duplicates()

    X = df[cfg['features']]
    y = df[cfg['target']]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

    modelo = LinearRegression().fit(X_train, y_train)
    pred = modelo.predict(X_test)

    plt.figure(figsize=(12, 4))
    for i, feature in enumerate(cfg['features']):
        plt.subplot(1, len(cfg['features']), i + 1)
        plt.scatter(df[feature], df[cfg['target']], alpha=0.5, color='#2a6f97')
        m, b = np.polyfit(df[feature], df[cfg['target']], 1)
        xs = np.linspace(df[feature].min(), df[feature].max(), 100)
        plt.plot(xs, m * xs + b, color='#c8553d', lw=2)
        plt.title(f'{feature} vs {cfg[target]}')
        plt.xlabel(feature)
        plt.ylabel(cfg['target'])
    plt.suptitle(cfg['titulo'], fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 5))
    corr = df[cfg['features'] + [cfg['target']]].corr()
    plt.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=30, ha='right')
    plt.yticks(range(len(corr.columns)), corr.columns)
    for i in range(len(corr.columns)):
        for j in range(len(corr.columns)):
            plt.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')
    plt.colorbar(label='Correlación')
    plt.title(f'Matriz de correlación - {cfg[titulo]}')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 5))
    plt.scatter(y_test, pred, alpha=0.6, color='#2a6f97')
    minv, maxv = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
    plt.plot([minv, maxv], [minv, maxv], color='#c8553d', ls='--', lw=2)
    plt.xlabel('Valor real')
    plt.ylabel('Valor predicho')
    plt.title(f'Real vs predicho - R² = {r2_score(y_test, pred):.3f}')
    plt.tight_layout()
    plt.show()

## Conclusión

El mejor modelo se puede identificar con el valor más alto de R² y el menor RMSE.

- El modelo de dólar tuvo excelente ajuste.
- El de glucosa tuvo un desempeño moderado.
- El de energía mostró buen desempeño en consumo y temperatura/hora.

Este notebook es ideal para subir a Google Colab o presentarlo en clase, porque se ejecuta con los datos originales y genera un dashboard visual de resultados.